
# Flight Delay ML Platform — Baseline Modeling

## 1. Modeling Objectives and Evaluation Protocol

### Prediction Tasks

Two independent binary classifiers will be developed:

1. Departure delay prediction: `DepDel15`
2. Arrival delay prediction: `ArrDel15`

A positive label indicates a delay of at least 15 minutes.

### Data Splits

| Split | Period | Purpose |
|---|---|---|
| Train | 2023 | Model fitting |
| Validation | Jan–Aug 2024 | Model selection |
| Calibration | Sep–Oct 2024 | Probability calibration |
| Threshold | Nov–Dec 2024 | Decision threshold selection |
| Test | 2025 | Final out-of-time evaluation |

### Primary Evaluation Metric

Average Precision (AP), commonly reported as PR-AUC, is the primary metric.

Additional metrics include:

- ROC-AUC
- Accuracy
- Precision
- Recall
- F1 Score
- Brier Score

Average Precision is used because it summarizes the precision-recall tradeoff across prediction thresholds.

### Baseline Models

1. Dummy Classifier
2. Logistic Regression

More advanced models will be evaluated after establishing reproducible baselines.

### Leakage Prevention

Model fitting uses only the 2023 training data.

The 2024 validation data is used for model evaluation and selection.

Calibration and threshold datasets remain separate, and the 2025 final test set remains untouched during development.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from flight_delay.data.dataset import load_temporal_split

# Locate project root.
current = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "pyproject.toml").exists()
    ),
    None,
)

assert ROOT is not None

processed_dir = ROOT / "data" / "processed"

# Load training and validation splits only.
train = load_temporal_split(
    "train",
    processed_dir=processed_dir,
)

validation = load_temporal_split(
    "validation",
    processed_dir=processed_dir,
)

print("=== Dataset Summary ===")
print("Train:", train.X.shape)
print("Validation:", validation.X.shape)

assert train.X.shape == (1_478_412, 17)
assert validation.X.shape == (1_020_082, 17)

assert train.flight_dates.max() < validation.flight_dates.min()

print("\nChronological split verification passed!")

In [ ]:
from sklearn.dummy import DummyClassifier

dummy_models = {}

targets = {
    "Departure": (
        train.y_departure,
        validation.y_departure,
    ),
    "Arrival": (
        train.y_arrival,
        validation.y_arrival,
    ),
}

# Dummy models do not use actual flight features.
X_train_dummy = np.zeros(
    (len(train.X), 1),
    dtype=np.uint8,
)

X_val_dummy = np.zeros(
    (len(validation.X), 1),
    dtype=np.uint8,
)

for task, (y_train, y_val) in targets.items():

    model = DummyClassifier(strategy="prior")

    model.fit(X_train_dummy, y_train)

    dummy_models[task] = model

    print(f"=== {task} Dummy Classifier ===")
    print("Training positive rate:", f"{y_train.mean():.2%}")
    print("Validation positive rate:", f"{y_val.mean():.2%}")
    print("Learned classes:", model.classes_)
    print()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

dummy_results = []

for task, (_, y_val) in targets.items():
    model = dummy_models[task]

    # Probability of positive class (delay >= 15 minutes).
    positive_idx = list(model.classes_).index(1)

    y_prob = model.predict_proba(
        X_val_dummy
    )[:, positive_idx]

    y_pred = model.predict(X_val_dummy)

    dummy_results.append({
        "Task": task,
        "Model": "Dummy (Prior)",
        "AP": average_precision_score(y_val, y_prob),
        "ROC_AUC": roc_auc_score(y_val, y_prob),
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(
            y_val, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, y_pred, zero_division=0
        ),
        "Brier": brier_score_loss(y_val, y_prob),
    })

dummy_report = pd.DataFrame(dummy_results)

display(dummy_report.round(4))

print("\nDummy baseline evaluation passed!")


## 3. Logistic Regression Baseline

### Objective

Train the first feature-based classification model for departure delay prediction.

The model predicts whether a flight will depart at least 15 minutes late (`DepDel15 = 1`).

### Model Architecture

A Scikit-learn Pipeline combines:

1. One-Hot Encoding for categorical variables
2. Standard Scaling for numerical variables
3. Logistic Regression for binary classification

### Training Configuration

- Training period: January–December 2023
- Validation period: January–August 2024
- Solver: L-BFGS
- Regularization: L2
- Regularization strength: C = 1.0
- Maximum iterations: 300
- Class weighting: None

### Evaluation Strategy

The primary metric is Average Precision (AP).

Secondary metrics include ROC-AUC, Accuracy, Precision, Recall, F1, and Brier Score.

The model is compared against the Dummy Classifier baseline.

### Leakage Prevention

The entire Pipeline is fitted using only the training split.

The validation split is used exclusively for evaluation.

No probability calibration or decision threshold optimization is performed at this stage.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from flight_delay.models.preprocessing import (
    make_model_preprocessor,
    validate_model_features,
)

# Validate input feature schemas.
validate_model_features(train.X)
validate_model_features(validation.X)

# Build the complete modeling pipeline.
lr_departure = Pipeline(
    steps=[
        (
            "preprocessor",
            make_model_preprocessor(),
        ),
        (
            "classifier",
            LogisticRegression(
                solver="lbfgs",
                penalty="l2",
                C=1.0,
                max_iter=300,
                class_weight=None,
            ),
        ),
    ]
)

print("Training Departure Logistic Regression...")

# Fit preprocessing AND classifier on 2023 data only.
lr_departure.fit(
    train.X,
    train.y_departure,
)

print("\nTraining completed!")

# Inspect the fitted pipeline.
encoded_features = (
    lr_departure
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

classifier = lr_departure.named_steps["classifier"]

print("Encoded feature count:", len(encoded_features))
print("Model classes:", classifier.classes_)
print("Iterations:", classifier.n_iter_)
print("Coefficient shape:", classifier.coef_.shape)

assert len(encoded_features) == 312
assert classifier.coef_.shape == (1, 312)
assert classifier.classes_.tolist() == [0, 1]

print("\nDeparture Logistic Regression verification passed!")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Predict on validation data without refitting.
y_val_dep = validation.y_departure

dep_prob = lr_departure.predict_proba(
    validation.X
)[:, 1]

# Default classification threshold.
dep_pred = (dep_prob >= 0.5).astype("int8")

# Calculate evaluation metrics.
lr_dep_result = {
    "Task": "Departure",
    "Model": "Logistic Regression",
    "AP": average_precision_score(
        y_val_dep, dep_prob
    ),
    "ROC_AUC": roc_auc_score(
        y_val_dep, dep_prob
    ),
    "Accuracy": accuracy_score(
        y_val_dep, dep_pred
    ),
    "Precision": precision_score(
        y_val_dep, dep_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_val_dep, dep_pred, zero_division=0
    ),
    "F1": f1_score(
        y_val_dep, dep_pred, zero_division=0
    ),
    "Brier": brier_score_loss(
        y_val_dep, dep_prob
    ),
}

lr_dep_report = pd.DataFrame([lr_dep_result])

print("=== Departure Logistic Regression ===")
display(lr_dep_report.round(4))

print("Validation positive rate:", f"{y_val_dep.mean():.2%}")
print("Mean predicted probability:", f"{dep_prob.mean():.2%}")

assert len(dep_prob) == len(y_val_dep)
assert np.isfinite(dep_prob).all()
assert ((dep_prob >= 0) & (dep_prob <= 1)).all()

print("\nDeparture model evaluation passed!")

In [ ]:
dummy_dep = dummy_report.loc[
    dummy_report["Task"] == "Departure"
].copy()

comparison_dep = pd.concat(
    [
        dummy_dep,
        lr_dep_report,
    ],
    ignore_index=True,
)

metrics = [
    "AP",
    "ROC_AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Brier",
]

print("=== Departure Model Comparison ===")
display(
    comparison_dep[
        ["Model", *metrics]
    ].round(4)
)

dummy_ap = float(dummy_dep["AP"].iloc[0])
lr_ap = float(lr_dep_report["AP"].iloc[0])

print("Dummy AP:", round(dummy_ap, 4))
print("Logistic Regression AP:", round(lr_ap, 4))
print("Absolute AP improvement:", round(lr_ap - dummy_ap, 4))

print("\nBaseline comparison completed!")


## 4. Arrival Delay Logistic Regression

### Objective

Train a separate Logistic Regression classifier to predict whether a flight will arrive at least 15 minutes late.

The arrival model uses the same 17 engineered features and chronological data splits as the departure model.

### Training Configuration

- Target: `ArrDel15`
- Training period: January–December 2023
- Validation period: January–August 2024
- Regularization: L2
- C: 1.0
- Solver: L-BFGS
- Maximum iterations: 300
- Class weighting: None

### Evaluation Strategy

The model is evaluated using Average Precision, ROC-AUC, Accuracy, Precision, Recall, F1, and Brier Score.

The primary comparison is against the Arrival Dummy Classifier.

Both Logistic Regression models use the same feature engineering and preprocessing configurations, but are fitted independently for their respective targets.

No probability calibration or threshold optimization is performed during this experiment.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from flight_delay.models.preprocessing import (
    make_model_preprocessor,
    validate_model_features,
)

# Verify feature schemas.
validate_model_features(train.X)
validate_model_features(validation.X)

# Independent model for arrival delay prediction.
lr_arrival = Pipeline(
    steps=[
        (
            "preprocessor",
            make_model_preprocessor(),
        ),
        (
            "classifier",
            LogisticRegression(
                solver="lbfgs",
                penalty="l2",
                C=1.0,
                max_iter=300,
                class_weight=None,
            ),
        ),
    ]
)

print("Training Arrival Logistic Regression...")

lr_arrival.fit(
    train.X,
    train.y_arrival,
)

classifier_arr = lr_arrival.named_steps["classifier"]

encoded_features_arr = (
    lr_arrival.named_steps["preprocessor"]
    .get_feature_names_out()
)

print("\n=== Arrival Model Training ===")
print("Encoded features:", len(encoded_features_arr))
print("Model classes:", classifier_arr.classes_)
print("Iterations:", classifier_arr.n_iter_)
print("Coefficient shape:", classifier_arr.coef_.shape)

assert classifier_arr.classes_.tolist() == [0, 1]
assert classifier_arr.coef_.shape == (1, 312)

print("\nArrival Logistic Regression training passed!")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

y_val_arr = validation.y_arrival

# Predict arrival delay probabilities.
arr_prob = lr_arrival.predict_proba(
    validation.X
)[:, 1]

# Use default 0.5 threshold for baseline evaluation.
arr_pred = (arr_prob >= 0.5).astype("int8")

lr_arr_result = {
    "Task": "Arrival",
    "Model": "Logistic Regression",
    "AP": average_precision_score(y_val_arr, arr_prob),
    "ROC_AUC": roc_auc_score(y_val_arr, arr_prob),
    "Accuracy": accuracy_score(y_val_arr, arr_pred),
    "Precision": precision_score(
        y_val_arr, arr_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_val_arr, arr_pred, zero_division=0
    ),
    "F1": f1_score(
        y_val_arr, arr_pred, zero_division=0
    ),
    "Brier": brier_score_loss(y_val_arr, arr_prob),
}

lr_arr_report = pd.DataFrame([lr_arr_result])

print("=== Arrival Logistic Regression ===")
display(lr_arr_report.round(4))

print(
    "Validation positive rate:",
    f"{y_val_arr.mean():.2%}"
)
print(
    "Mean predicted probability:",
    f"{arr_prob.mean():.2%}"
)

assert len(arr_prob) == len(y_val_arr)
assert np.isfinite(arr_prob).all()
assert ((arr_prob >= 0) & (arr_prob <= 1)).all()

print("\nArrival model evaluation passed!")

In [ ]:
# Combine Dummy and Logistic Regression results.
all_baseline_results = pd.concat(
    [
        dummy_report,
        pd.DataFrame([lr_dep_result, lr_arr_result]),
    ],
    ignore_index=True,
)

metrics = [
    "AP",
    "ROC_AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Brier",
]

print("=== Baseline Model Comparison ===")

display(
    all_baseline_results[
        ["Task", "Model", *metrics]
    ].round(4)
)

# Calculate AP improvement for each task.
for task in ["Departure", "Arrival"]:
    subset = all_baseline_results[
        all_baseline_results["Task"] == task
    ]

    dummy_ap = subset.loc[
        subset["Model"] == "Dummy (Prior)", "AP"
    ].iloc[0]

    lr_ap = subset.loc[
        subset["Model"] == "Logistic Regression", "AP"
    ].iloc[0]

    print(f"\n{task} AP Improvement")
    print(f"Dummy AP: {dummy_ap:.4f}")
    print(f"Logistic Regression AP: {lr_ap:.4f}")
    print(f"Absolute improvement: {lr_ap - dummy_ap:+.4f}")
    print(f"Relative improvement: {(lr_ap / dummy_ap - 1):+.2%}")

## 5. Precision-Recall Curve Analysis

### Objective

Evaluate how precision changes with recall across classification thresholds.

### Motivation

Both Logistic Regression models outperform their Dummy baselines in Average Precision, but recall at the default 0.5 threshold is very low.

Precision-recall curves provide a threshold-independent view of the tradeoff between identifying delayed flights and producing false alarms.

### Evaluation Protocol

- Dataset: January–August 2024 validation
- Models: Departure and Arrival Logistic Regression
- Primary metric: Average Precision
- Reference: Validation positive-class prevalence

Classification thresholds will not be optimized using this validation dataset.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
)

pr_tasks = {
    "Departure": {
        "y_true": validation.y_departure,
        "y_prob": dep_prob,
    },
    "Arrival": {
        "y_true": validation.y_arrival,
        "y_prob": arr_prob,
    },
}

for task, data in pr_tasks.items():

    y_true = data["y_true"]
    y_prob = data["y_prob"]

    precision, recall, thresholds = (
        precision_recall_curve(y_true, y_prob)
    )

    ap = average_precision_score(y_true, y_prob)
    prevalence = y_true.mean()

    plt.figure(figsize=(7, 5))

    plt.plot(
        recall,
        precision,
        label=f"Logistic Regression (AP = {ap:.4f})",
    )

    plt.axhline(
        y=prevalence,
        linestyle="--",
        label=f"Baseline prevalence = {prevalence:.4f}",
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{task} Delay — Precision-Recall Curve")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    print(f"{task} AP: {ap:.4f}")
    print(f"{task} prevalence: {prevalence:.4f}")
    print()

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)

monthly_model_results = []

for task, data in pr_tasks.items():

    audit_df = pd.DataFrame({
        "Month": validation.flight_dates.dt.to_period(
            "M"
        ).astype(str),
        "Actual": data["y_true"].to_numpy(),
        "PredictedProbability": data["y_prob"],
    })

    for month, group in audit_df.groupby(
        "Month", sort=True
    ):

        y_true = group["Actual"]
        y_prob = group["PredictedProbability"]

        monthly_model_results.append({
            "Task": task,
            "Month": month,
            "Records": len(group),
            "PositiveRate": y_true.mean(),
            "AP": average_precision_score(
                y_true, y_prob
            ),
            "ROC_AUC": roc_auc_score(
                y_true, y_prob
            ),
            "Brier": brier_score_loss(
                y_true, y_prob
            ),
            "MeanPredictedProbability": y_prob.mean(),
        })

monthly_model_report = pd.DataFrame(
    monthly_model_results
)

print("=== Monthly Model Performance ===")

display(
    monthly_model_report.round(4)
)

assert len(monthly_model_report) == 16

print("\nMonthly evaluation completed!")

In [ ]:
monthly_diagnostic = monthly_model_report.copy()

# AP above each month's positive-class prevalence.
monthly_diagnostic["AP_Above_Prevalence"] = (
    monthly_diagnostic["AP"]
    - monthly_diagnostic["PositiveRate"]
)

# Aggregate probability bias in percentage points.
monthly_diagnostic["Probability_Gap_pp"] = (
    monthly_diagnostic["MeanPredictedProbability"]
    - monthly_diagnostic["PositiveRate"]
) * 100

display(
    monthly_diagnostic[
        [
            "Task",
            "Month",
            "PositiveRate",
            "AP",
            "AP_Above_Prevalence",
            "ROC_AUC",
            "Probability_Gap_pp",
        ]
    ].round(4)
)


### Monthly Validation Performance — Results and Discussion

The Logistic Regression models were evaluated separately for each month from January through August 2024.

Both models demonstrated predictive value across the validation period, but their performance varied by month.

**Ranking Performance**

January showed the weakest ROC-AUC for both prediction tasks:

- Departure: 0.6060
- Arrival: 0.5746

In later months, ROC-AUC was generally higher, suggesting that the models were more effective at distinguishing delayed and non-delayed flights during those periods.

**Seasonal Variation in Delay Prevalence**

Delay prevalence varied considerably throughout the validation period. For example, departure delay prevalence increased from 17.34% in February to 31.49% in July.

Because Average Precision depends on positive-class prevalence, monthly AP values should not be compared without considering the corresponding baseline rates.

**Aggregate Probability Bias**

The models also showed month-dependent differences between predicted and observed delay rates.

In July, the departure model underestimated the observed delay rate by approximately 5.02 percentage points, while the arrival model underestimated it by approximately 5.78 percentage points.

In February, both models overestimated the observed delay rates by more than 4 percentage points.

These findings suggest possible temporal instability in probability estimates. However, aggregate probability bias alone does not establish the full calibration behavior of either model.

**Implications**

Future experiments should investigate whether nonlinear models can improve ranking performance and reduce temporal variation.

Probability calibration will be evaluated separately using the reserved calibration dataset.

The 2025 test dataset remains untouched.



## 6. LightGBM Nonlinear Baseline

### Objective

Evaluate whether a nonlinear gradient boosting model improves flight delay prediction using the same 17 engineered features as Logistic Regression.

### Experimental Design

- Training: January–December 2023
- Validation: January–August 2024
- Features: Same 17 scheduled-flight and calendar features
- Targets: Departure and arrival delays, modeled independently
- Primary metric: Average Precision

### Feature Representation

Unlike Logistic Regression, LightGBM uses native categorical features instead of one-hot encoding.

Categorical levels are learned exclusively from the training dataset and reused for validation.

Numerical features are retained without standard scaling.

### Initial Model Configuration

A fixed LightGBM configuration will be used to establish the first nonlinear baseline.

No hyperparameter tuning, early stopping, probability calibration, or threshold optimization is performed in this experiment.

The 2025 final test dataset remains untouched.


In [ ]:
from flight_delay.models.lightgbm_baseline import (
    fit_category_levels,
    prepare_lightgbm_features,
)

# Learn category definitions from training data only.
category_levels = fit_category_levels(train.X)

# Prepare training and validation features.
X_train_lgb = prepare_lightgbm_features(
    train.X,
    category_levels,
)

X_val_lgb = prepare_lightgbm_features(
    validation.X,
    category_levels,
)

print("=== LightGBM Feature Preparation ===")
print("Training shape:", X_train_lgb.shape)
print("Validation shape:", X_val_lgb.shape)

print("\nCategorical features:")
print(
    X_train_lgb.select_dtypes(
        include="category"
    ).columns.tolist()
)

print("\nCategory counts:")
for col, levels in category_levels.items():
    print(f"{col}: {len(levels)}")

assert X_train_lgb.shape == (1_478_412, 17)
assert X_val_lgb.shape == (1_020_082, 17)

print("\nLightGBM feature preparation passed!")

In [ ]:
import lightgbm as lgb

lgb_departure = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=100,
    subsample=0.9,
    subsample_freq=1,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=4,
    verbosity=-1,
)

print("Training Departure LightGBM...")

lgb_departure.fit(
    X_train_lgb,
    train.y_departure,
    categorical_feature=list(category_levels),
)

print("\n=== Departure LightGBM Training ===")
print("Number of features:", lgb_departure.n_features_in_)
print("Number of trees:", lgb_departure.n_estimators_)
print("Model classes:", lgb_departure.classes_)

assert lgb_departure.n_features_in_ == 17
assert lgb_departure.classes_.tolist() == [0, 1]

print("\nDeparture LightGBM training passed!")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

y_val_dep = validation.y_departure

# Positive-class probabilities.
lgb_dep_prob = lgb_departure.predict_proba(
    X_val_lgb
)[:, 1]

# Default threshold, for comparison only.
lgb_dep_pred = (
    lgb_dep_prob >= 0.5
).astype("int8")

lgb_dep_result = {
    "Task": "Departure",
    "Model": "LightGBM",
    "AP": average_precision_score(
        y_val_dep, lgb_dep_prob
    ),
    "ROC_AUC": roc_auc_score(
        y_val_dep, lgb_dep_prob
    ),
    "Accuracy": accuracy_score(
        y_val_dep, lgb_dep_pred
    ),
    "Precision": precision_score(
        y_val_dep, lgb_dep_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_val_dep, lgb_dep_pred, zero_division=0
    ),
    "F1": f1_score(
        y_val_dep, lgb_dep_pred, zero_division=0
    ),
    "Brier": brier_score_loss(
        y_val_dep, lgb_dep_prob
    ),
}

lgb_dep_report = pd.DataFrame([lgb_dep_result])

print("=== Departure LightGBM Results ===")
display(lgb_dep_report.round(4))

print(
    "Mean predicted probability:",
    f"{lgb_dep_prob.mean():.2%}"
)

assert len(lgb_dep_prob) == len(y_val_dep)
assert np.isfinite(lgb_dep_prob).all()
assert ((lgb_dep_prob >= 0) & (lgb_dep_prob <= 1)).all()

print("\nDeparture LightGBM evaluation passed!")

In [ ]:
departure_comparison = pd.concat(
    [
        dummy_report.loc[
            dummy_report["Task"] == "Departure"
        ],
        lr_dep_report,
        lgb_dep_report,
    ],
    ignore_index=True,
)

print("=== Departure Model Comparison ===")

display(
    departure_comparison[
        [
            "Model",
            "AP",
            "ROC_AUC",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "Brier",
        ]
    ].round(4)
)

lr_ap = lr_dep_result["AP"]
lgb_ap = lgb_dep_result["AP"]

print(f"Logistic Regression AP: {lr_ap:.4f}")
print(f"LightGBM AP: {lgb_ap:.4f}")
print(f"AP improvement: {lgb_ap - lr_ap:+.4f}")
print(f"Relative AP improvement: {(lgb_ap / lr_ap - 1):+.2%}")

In [ ]:
import lightgbm as lgb

# Same hyperparameters as Departure LightGBM.
lgb_arrival = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=100,
    subsample=0.9,
    subsample_freq=1,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=4,
    verbosity=-1,
)

print("Training Arrival LightGBM...")

lgb_arrival.fit(
    X_train_lgb,
    train.y_arrival,
    categorical_feature=list(category_levels),
)

print("\n=== Arrival LightGBM Training ===")
print("Number of features:", lgb_arrival.n_features_in_)
print("Number of trees:", lgb_arrival.n_estimators_)
print("Model classes:", lgb_arrival.classes_)

assert lgb_arrival.n_features_in_ == 17
assert lgb_arrival.classes_.tolist() == [0, 1]

print("\nArrival LightGBM training passed!")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

y_val_arr = validation.y_arrival

lgb_arr_prob = lgb_arrival.predict_proba(
    X_val_lgb
)[:, 1]

lgb_arr_pred = (
    lgb_arr_prob >= 0.5
).astype("int8")

lgb_arr_result = {
    "Task": "Arrival",
    "Model": "LightGBM",
    "AP": average_precision_score(
        y_val_arr, lgb_arr_prob
    ),
    "ROC_AUC": roc_auc_score(
        y_val_arr, lgb_arr_prob
    ),
    "Accuracy": accuracy_score(
        y_val_arr, lgb_arr_pred
    ),
    "Precision": precision_score(
        y_val_arr, lgb_arr_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_val_arr, lgb_arr_pred, zero_division=0
    ),
    "F1": f1_score(
        y_val_arr, lgb_arr_pred, zero_division=0
    ),
    "Brier": brier_score_loss(
        y_val_arr, lgb_arr_prob
    ),
}

lgb_arr_report = pd.DataFrame([lgb_arr_result])

print("=== Arrival LightGBM Results ===")
display(lgb_arr_report.round(4))

print(
    "Mean predicted probability:",
    f"{lgb_arr_prob.mean():.2%}"
)

assert len(lgb_arr_prob) == len(y_val_arr)
assert np.isfinite(lgb_arr_prob).all()
assert ((lgb_arr_prob >= 0) & (lgb_arr_prob <= 1)).all()

print("\nArrival LightGBM evaluation passed!")

In [ ]:
baseline_comparison = pd.concat(
    [
        dummy_report,
        lr_dep_report,
        lr_arr_report,
        lgb_dep_report,
        lgb_arr_report,
    ],
    ignore_index=True,
)

baseline_comparison = baseline_comparison.sort_values(
    ["Task", "Model"]
).reset_index(drop=True)

print("=== All Baseline Models ===")

display(
    baseline_comparison[
        [
            "Task",
            "Model",
            "AP",
            "ROC_AUC",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "Brier",
        ]
    ].round(4)
)

print("\n=== LightGBM vs Logistic Regression ===")

for task in ["Departure", "Arrival"]:
    subset = baseline_comparison[
        baseline_comparison["Task"] == task
    ]

    lr_ap = subset.loc[
        subset["Model"] == "Logistic Regression",
        "AP"
    ].iloc[0]

    lgb_ap = subset.loc[
        subset["Model"] == "LightGBM",
        "AP"
    ].iloc[0]

    print(f"\n{task}")
    print(f"Logistic Regression AP: {lr_ap:.4f}")
    print(f"LightGBM AP: {lgb_ap:.4f}")
    print(f"AP difference: {lgb_ap - lr_ap:+.4f}")


## 7. LightGBM vs. Logistic Regression: Temporal Robustness

### Objective

Compare Logistic Regression and LightGBM across individual validation months to identify temporal variation in model performance.

### Evaluation Period

January–August 2024, using the original chronological validation split.

### Evaluation Metrics

- Average Precision (AP)
- ROC-AUC
- Brier Score
- Mean Predicted Probability
- Observed Delay Prevalence

### Research Questions

1. Does LightGBM consistently outperform Logistic Regression across months?
2. Does LightGBM improve the relatively weak January ranking performance?
3. Does LightGBM reduce the aggregate probability bias observed in February and July?

### Methodology

Both models use the same validation records and scheduled-flight feature information.

No additional model fitting, hyperparameter optimization, calibration, or threshold selection is performed during this analysis.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)

# Use the same monthly groups as Logistic Regression.
validation_months = (
    validation.flight_dates
    .dt.to_period("M")
    .astype(str)
    .to_numpy()
)

lgb_tasks = {
    "Departure": (
        validation.y_departure,
        lgb_dep_prob,
    ),
    "Arrival": (
        validation.y_arrival,
        lgb_arr_prob,
    ),
}

lgb_monthly_results = []

for task, (y_true, y_prob) in lgb_tasks.items():

    eval_df = pd.DataFrame({
        "Month": validation_months,
        "Actual": np.asarray(y_true),
        "PredictedProbability": np.asarray(y_prob),
    })

    for month, group in eval_df.groupby(
        "Month", sort=True
    ):

        actual = group["Actual"]
        probability = group["PredictedProbability"]

        lgb_monthly_results.append({
            "Task": task,
            "Month": month,
            "Records": len(group),
            "PositiveRate": actual.mean(),
            "AP": average_precision_score(
                actual, probability
            ),
            "ROC_AUC": roc_auc_score(
                actual, probability
            ),
            "Brier": brier_score_loss(
                actual, probability
            ),
            "MeanPredictedProbability": probability.mean(),
        })

lgb_monthly_report = pd.DataFrame(
    lgb_monthly_results
)

display(lgb_monthly_report.round(4))

assert len(lgb_monthly_report) == 16

print("\nLightGBM monthly evaluation passed!")

In [ ]:
monthly_comparison = monthly_model_report.merge(
    lgb_monthly_report,
    on=["Task", "Month"],
    suffixes=("_LR", "_LGB"),
    validate="one_to_one",
)

# Confirm both models evaluated identical samples.
assert (
    monthly_comparison["Records_LR"]
    == monthly_comparison["Records_LGB"]
).all()

np.testing.assert_allclose(
    monthly_comparison["PositiveRate_LR"],
    monthly_comparison["PositiveRate_LGB"],
)

# Positive delta: improvement for AP / ROC-AUC.
monthly_comparison["Delta_AP"] = (
    monthly_comparison["AP_LGB"]
    - monthly_comparison["AP_LR"]
)

monthly_comparison["Delta_ROC_AUC"] = (
    monthly_comparison["ROC_AUC_LGB"]
    - monthly_comparison["ROC_AUC_LR"]
)

# Negative delta: improvement for Brier.
monthly_comparison["Delta_Brier"] = (
    monthly_comparison["Brier_LGB"]
    - monthly_comparison["Brier_LR"]
)

print("=== Monthly Model Comparison ===")

display(
    monthly_comparison[
        [
            "Task",
            "Month",
            "AP_LR",
            "AP_LGB",
            "Delta_AP",
            "ROC_AUC_LR",
            "ROC_AUC_LGB",
            "Delta_ROC_AUC",
            "Brier_LR",
            "Brier_LGB",
            "Delta_Brier",
        ]
    ].round(4)
)

print("\nMonthly model comparison completed!")

In [ ]:
monthly_comparison["LR_Probability_Gap_pp"] = (
    monthly_comparison["MeanPredictedProbability_LR"]
    - monthly_comparison["PositiveRate_LR"]
) * 100

monthly_comparison["LGB_Probability_Gap_pp"] = (
    monthly_comparison["MeanPredictedProbability_LGB"]
    - monthly_comparison["PositiveRate_LGB"]
) * 100

print("=== Monthly Aggregate Probability Bias ===")

display(
    monthly_comparison[
        [
            "Task",
            "Month",
            "LR_Probability_Gap_pp",
            "LGB_Probability_Gap_pp",
        ]
    ].round(3)
)

print("\nProbability bias comparison completed!")


## 8. LightGBM Temporal Robustness Investigation

### Problem

LightGBM exhibited substantially weaker ranking performance in January 2024 compared with Logistic Regression, especially for arrival delays.

However, LightGBM performed better in several later months.

### Investigation Objectives

1. Verify whether validation data contains unseen categorical values.
2. Check whether poor January performance is concentrated at specific origin airports.
3. Identify potential temporal distribution shifts before introducing additional features or tuning hyperparameters.

The analysis uses the existing 2024 validation data only. No models are refitted.


In [ ]:
import numpy as np
import pandas as pd

validation_months = (
    validation.flight_dates
    .dt.to_period("M")
    .astype(str)
    .to_numpy()
)

category_audit = []

for col in category_levels:

    original = validation.X[col].astype("string")
    encoded = X_val_lgb[col]

    # A known raw value converted to NaN indicates
    # a category unseen during training.
    is_unknown = (
        original.notna().to_numpy()
        & encoded.isna().to_numpy()
    )

    for month in sorted(np.unique(validation_months)):
        mask = validation_months == month

        category_audit.append({
            "Feature": col,
            "Month": month,
            "Records": int(mask.sum()),
            "UnknownCount": int(is_unknown[mask].sum()),
            "UnknownRate": float(is_unknown[mask].mean()),
        })

category_audit_df = pd.DataFrame(category_audit)

print("=== Unknown Category Audit ===")

display(
    category_audit_df.round(5)
)

print("\nCategorical integrity audit completed!")

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

arrival_audit = pd.DataFrame({
    "Month": validation_months,
    "Origin": validation.X["Origin"].to_numpy(),
    "Actual": validation.y_arrival.to_numpy(),
    "LR_Prob": arr_prob,
    "LGB_Prob": lgb_arr_prob,
})

airport_results = []

for month in ["2024-01", "2024-08"]:

    month_data = arrival_audit[
        arrival_audit["Month"] == month
    ]

    for origin, group in month_data.groupby("Origin"):

        y_true = group["Actual"]

        # ROC-AUC requires both classes.
        if y_true.nunique() < 2:
            continue

        lr_auc = roc_auc_score(
            y_true,
            group["LR_Prob"],
        )

        lgb_auc = roc_auc_score(
            y_true,
            group["LGB_Prob"],
        )

        airport_results.append({
            "Month": month,
            "Origin": origin,
            "Records": len(group),
            "PositiveRate": y_true.mean(),
            "LR_AP": average_precision_score(
                y_true, group["LR_Prob"]
            ),
            "LGB_AP": average_precision_score(
                y_true, group["LGB_Prob"]
            ),
            "LR_ROC_AUC": lr_auc,
            "LGB_ROC_AUC": lgb_auc,
            "Delta_ROC_AUC": lgb_auc - lr_auc,
        })

airport_report = pd.DataFrame(airport_results)

print("=== Arrival Performance by Origin Airport ===")

display(
    airport_report.sort_values(
        ["Month", "Delta_ROC_AUC"]
    ).round(4)
)

print("\nAirport-level diagnosis completed!")


## 9. LightGBM Feature Importance Analysis

### Objective

Identify which scheduled-flight and calendar features contribute most to the fitted LightGBM models.

### Method

Feature importance is calculated using the total split gain from each trained LightGBM model.

Gain measures the total improvement in the model's tree-splitting objective associated with a feature.

For easier comparison, gain values are normalized to percentages within each model.

### Research Questions

1. Which features are most important for departure delay prediction?
2. Which features are most important for arrival delay prediction?
3. Are the two models relying on similar feature groups?
4. Do the results suggest hypotheses worth investigating about temporal instability?

### Limitations

Global feature importance does not establish causality.

Importance can be affected by correlated predictors, feature cardinality, and the fitted model.

Global importance also does not directly explain the January 2024 performance degradation.


In [ ]:
import numpy as np
import pandas as pd

lgb_models = {
    "Departure": lgb_departure,
    "Arrival": lgb_arrival,
}

importance_results = []

for task, model in lgb_models.items():

    booster = model.booster_

    feature_names = booster.feature_name()

    gain_values = booster.feature_importance(
        importance_type="gain"
    )

    total_gain = gain_values.sum()

    assert total_gain > 0

    for feature, gain in zip(
        feature_names, gain_values, strict=True
    ):
        importance_results.append({
            "Task": task,
            "Feature": feature,
            "Gain": gain,
            "GainPercent": 100 * gain / total_gain,
        })

importance_df = pd.DataFrame(importance_results)

assert len(importance_df) == 34

for task in ["Departure", "Arrival"]:

    print(f"\n=== {task} Feature Importance ===")

    result = (
        importance_df[
            importance_df["Task"] == task
        ]
        .sort_values(
            "GainPercent",
            ascending=False
        )
    )

    display(
        result[
            ["Feature", "GainPercent"]
        ].head(10).round(2)
    )

print("\nFeature importance analysis completed!")

In [ ]:
import matplotlib.pyplot as plt

for task in ["Departure", "Arrival"]:

    top_features = (
        importance_df[
            importance_df["Task"] == task
        ]
        .nlargest(10, "GainPercent")
        .sort_values("GainPercent")
    )

    plt.figure(figsize=(8, 5))

    plt.barh(
        top_features["Feature"],
        top_features["GainPercent"],
    )

    plt.xlabel("Normalized Gain (%)")
    plt.title(f"{task} LightGBM — Top 10 Features")
    plt.tight_layout()
    plt.show()


## 10. January 2023 vs. January 2024 Distribution Shift

### Objective

Investigate whether the flight composition and delay prevalence changed between January 2023 and January 2024.

### Motivation

LightGBM exhibited significantly weaker ranking performance in January 2024, especially for arrival delay prediction at ORD, DEN, and DFW.

Feature importance analysis indicates that airline, airport, and scheduled-time features play important roles in the fitted models.

### Analysis

The investigation compares:

- Flight counts and delay prevalence
- Origin airport composition
- Airport-specific departure and arrival delay rates

### Limitations

January 2023 belongs to the training dataset, while January 2024 belongs to the validation dataset.

This analysis is descriptive and does not establish the cause of the observed model performance degradation.

The modeling cohort includes only completed, non-diverted flights with both delay labels available.


In [ ]:
import numpy as np
import pandas as pd

def make_january_audit(split, year):
    dates = split.flight_dates

    mask = (
        (dates.dt.year == year)
        & (dates.dt.month == 1)
    ).to_numpy()

    selected = split.X.iloc[np.flatnonzero(mask)]

    return pd.DataFrame({
        "Year": year,
        "Origin": selected["Origin"].to_numpy(),
        "Reporting_Airline": selected[
            "Reporting_Airline"
        ].to_numpy(),
        "Dest": selected["Dest"].to_numpy(),
        "DepartureDelay": np.asarray(
            split.y_departure
        )[mask],
        "ArrivalDelay": np.asarray(
            split.y_arrival
        )[mask],
    })


jan_2023 = make_january_audit(train, 2023)
jan_2024 = make_january_audit(validation, 2024)

january_comparison = pd.concat(
    [jan_2023, jan_2024],
    ignore_index=True,
)

# Overall January comparison.
january_summary = (
    january_comparison
    .groupby("Year")
    .agg(
        Flights=("ArrivalDelay", "size"),
        DepartureRate=("DepartureDelay", "mean"),
        ArrivalRate=("ArrivalDelay", "mean"),
    )
)

print("=== January 2023 vs. January 2024 ===")

display(january_summary.round(4))

print("\nJanuary distribution audit passed!")

In [ ]:
origin_summary = (
    january_comparison
    .groupby(["Year", "Origin"])
    .agg(
        Flights=("ArrivalDelay", "size"),
        DepartureRate=("DepartureDelay", "mean"),
        ArrivalRate=("ArrivalDelay", "mean"),
    )
    .reset_index()
)

# Calculate each airport's share of January flights.
origin_summary["FlightShare"] = (
    origin_summary["Flights"]
    / origin_summary.groupby("Year")["Flights"]
        .transform("sum")
)

# Compare the two years.
origin_rates = origin_summary.pivot(
    index="Origin",
    columns="Year",
    values="ArrivalRate",
)

origin_shares = origin_summary.pivot(
    index="Origin",
    columns="Year",
    values="FlightShare",
)

origin_shift = pd.DataFrame({
    "Origin": origin_rates.index,
    "ArrivalRate_2023": origin_rates[2023].values,
    "ArrivalRate_2024": origin_rates[2024].values,
    "ArrivalRateChange_pp": (
        origin_rates[2024].values
        - origin_rates[2023].values
    ) * 100,
    "FlightShareChange_pp": (
        origin_shares[2024].values
        - origin_shares[2023].values
    ) * 100,
})

print("=== January Origin-Level Distribution Shift ===")

display(
    origin_shift.sort_values(
        "ArrivalRateChange_pp",
        key=abs,
        ascending=False,
    ).round(4)
)

print("\nOrigin-level distribution comparison completed!")

### January Airline-Level Distribution Shift

The January comparison revealed substantial differences in arrival delay prevalence at DFW, DEN, and ORD between 2023 and 2024.

This analysis investigates changes in airline composition and airline-specific arrival delay rates within these airports.

For each airport-airline group, we compare:

- Number of flights in January 2023 and January 2024
- Arrival delay prevalence
- Airline share within the origin airport
- Year-over-year changes in delay prevalence and airline share

To reduce noise from very small groups, the displayed comparison includes only airline-airport combinations with at least 200 eligible flights in each January.

These results are descriptive and do not establish the cause of LightGBM's performance degradation.

In [ ]:
import pandas as pd

focus_origins = ["DFW", "DEN", "ORD"]

focus_data = january_comparison[
    january_comparison["Origin"].isin(focus_origins)
].copy()

# Summarize each airport-airline combination.
airline_summary = (
    focus_data
    .groupby(
        ["Origin", "Reporting_Airline", "Year"]
    )
    .agg(
        Flights=("ArrivalDelay", "size"),
        ArrivalRate=("ArrivalDelay", "mean"),
    )
    .reset_index()
)

# Share of flights within the origin airport.
airline_summary["AirportShare"] = (
    airline_summary["Flights"]
    / airline_summary.groupby(
        ["Origin", "Year"]
    )["Flights"].transform("sum")
)

# Compare both years side by side.
airline_shift = airline_summary.pivot(
    index=["Origin", "Reporting_Airline"],
    columns="Year",
    values=[
        "Flights",
        "ArrivalRate",
        "AirportShare",
    ],
)

airline_shift.columns = [
    f"{metric}_{year}"
    for metric, year in airline_shift.columns
]

airline_shift = airline_shift.reset_index()

# Focus on sufficiently represented groups
# in BOTH years.
airline_shift = airline_shift.dropna(
    subset=[
        "Flights_2023",
        "Flights_2024",
        "ArrivalRate_2023",
        "ArrivalRate_2024",
    ]
).copy()

airline_shift = airline_shift[
    (airline_shift["Flights_2023"] >= 200)
    & (airline_shift["Flights_2024"] >= 200)
].copy()

# Changes in percentage points.
airline_shift["RateChange_pp"] = (
    airline_shift["ArrivalRate_2024"]
    - airline_shift["ArrivalRate_2023"]
) * 100

airline_shift["ShareChange_pp"] = (
    airline_shift["AirportShare_2024"]
    - airline_shift["AirportShare_2023"]
) * 100

# Rank by absolute change in delay rate.
airline_shift["AbsRateChange"] = (
    airline_shift["RateChange_pp"].abs()
)

top_airline_changes = (
    airline_shift
    .sort_values(
        ["Origin", "AbsRateChange"],
        ascending=[True, False],
    )
    .groupby("Origin", group_keys=False)
    .head(5)
)

print("=== January Airline-Level Distribution Shift ===")

display(
    top_airline_changes[
        [
            "Origin",
            "Reporting_Airline",
            "Flights_2023",
            "Flights_2024",
            "ArrivalRate_2023",
            "ArrivalRate_2024",
            "RateChange_pp",
            "ShareChange_pp",
        ]
    ].round(4)
)

print("\nAirline distribution audit completed!")


### January 2024 Temporal Generalization Investigation — Summary

The initial LightGBM baseline showed substantially weaker ranking performance in January 2024, especially for arrival delay prediction at ORD, DEN, and DFW.

The categorical integrity audit found no unseen airline, origin, or destination categories in January 2024, making unknown-category handling an unlikely explanation.

Comparing January 2023 with January 2024 revealed that overall arrival delay prevalence changed only slightly, from 24.94% to 25.52%.

However, substantial differences appeared at the airport and airline levels.

At DFW, arrival delay rates increased by approximately 10.91 percentage points for AA and 9.01 percentage points for MQ. At DEN, several major airlines experienced decreases in delay prevalence. At ORD, both airline composition and airline-specific delay rates changed.

These findings demonstrate subgroup-level variation that is not apparent from aggregate prevalence alone.

The observed changes may be associated with temporal generalization challenges, but they do not establish why LightGBM performed poorly in January.

The investigation motivates future experiments involving historical operational features, temporal robustness, and feature ablation.

No model hyperparameters or decision thresholds were changed based on this diagnostic analysis.
